# Resume Parsing Pipeline

Extrai experiências profissionais estruturadas e computa embeddings de resumes brutos.

**Output:** `data/processed_candidates.json` com colunas `full_experience` e `embedding` (3072-dim).

## 0. Setup

In [ ]:
!pip install -q -U google-genai pydantic python-dotenv openai requests pandas

In [ ]:
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    !git clone https://github.com/Willblemos2/agent_workshop.git
    repo_root = Path('/content/agent_workshop')
    sys.path.insert(0, str(repo_root))
else:
    repo_root = Path(os.getcwd()).parents[1]
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / 'data'
DATA_DIR.mkdir(exist_ok=True)
print(f'repo_root : {repo_root}')
print(f'data dir  : {DATA_DIR}')

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
else:
    from src.shared.config import load_env
    load_env()

## 1. Provider Selection

In [ ]:
from src.shared.factory import build_resume_parser, build_embedding_provider

parser             = build_resume_parser()
embedding_provider = build_embedding_provider()

# ── Trocar provider manualmente ────────────────────────────────────────
# from src.applications.parsing.resume_parser import ResumeParser
# from src.foundation.llm import GeminiLLMProvider
# from src.foundation.embeddings import OpenAIEmbeddingProvider
# parser             = ResumeParser(GeminiLLMProvider(model='gemini-2.0-flash'))
# embedding_provider = OpenAIEmbeddingProvider(model='text-embedding-3-large')
# ───────────────────────────────────────────────────────────────────────

print(f'Parser LLM : {type(parser._llm).__name__}')
print(f'Embeddings : {type(embedding_provider).__name__}')

## 2. Dataset

`load_raw_candidates()` verifica `data/raw_candidates.json` primeiro.
Se não existir, baixa do Google Drive e pode ser salvo localmente para próximas execuções.

In [ ]:
from src.shared.dataset import load_raw_candidates

candidates_df = load_raw_candidates()

print(f'Candidatos: {len(candidates_df)}')
print(f'Colunas:    {list(candidates_df.columns)}')
candidates_df.head(2)

In [ ]:
# Opcional: salvar raw localmente para evitar download nas próximas execuções
import json
raw_local = DATA_DIR / 'raw_candidates.json'
if not raw_local.exists():
    with open(raw_local, 'w', encoding='utf-8') as f:
        json.dump(candidates_df.to_dict(orient='records'), f, ensure_ascii=False)
    print(f'Salvo em {raw_local}')
else:
    print(f'Já existe: {raw_local}')

## 3. Parse — Candidato Único

Teste em um resume antes de rodar o pipeline completo.

In [ ]:
sample = candidates_df.iloc[0]
print(f'Candidato: {sample["id"]}  ({len(sample["content"])} chars)')
print(sample['content'][:400], '...')

In [ ]:
experiences = parser.parse(sample['content'])

print(f'{len(experiences)} experiências:\n')
for exp in experiences:
    print(f'  [{exp.start_date} → {exp.end_date or "Current"}] {exp.job_title} @ {exp.company}')

## 4. Pipeline — Parse Completo

| Parâmetro | Descrição | Sugestão |
|---|---|---|
| `N_SAMPLE` | Quantos candidatos processar (`None` = todos) | `None` para dataset completo |
| `MAX_WORKERS` | Chamadas LLM em paralelo | `5` é seguro para gpt-4o-mini |
| `CHECKPOINT_PATH` | Arquivo para salvar progresso | Sempre usar — protege contra interrupções |

**Se interromper:** rode a célula novamente — candidatos já processados são pulados automaticamente.

In [ ]:
N_SAMPLE         = None   # None para processar todos
MAX_WORKERS      = 5
CHECKPOINT_PATH  = str(DATA_DIR / 'parse_checkpoint.json')
CHECKPOINT_EVERY = 20

df_to_process = candidates_df.head(N_SAMPLE) if N_SAMPLE else candidates_df
print(f'Processando {len(df_to_process)} candidatos com {MAX_WORKERS} workers')
print(f'Checkpoint : {CHECKPOINT_PATH}')

In [ ]:
processed_df = parser.enrich_dataframe(
    df_to_process,
    max_workers=MAX_WORKERS,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_every=CHECKPOINT_EVERY,
)

In [ ]:
success = processed_df['full_experience'].apply(lambda x: len(x) > 0).sum()
print(f'Parse — ok: {success}/{len(processed_df)}')
processed_df[['id', 'full_experience']].head(2)

## 5. Embeddings

Computa embeddings do campo `content` em chunks. A coluna `embedding` é recriada do zero.

> **Custo com text-embedding-3-large:** ~$0,13 / 1.000 resumes

In [ ]:
import time

CHUNK_SIZE = 50  # reduza para 10 se atingir rate limit

processed_df = processed_df.drop(columns=['embedding', 'embedded_content'], errors='ignore')

texts = processed_df['content'].tolist()
total = len(texts)
all_embeddings = []
start = time.time()

for i in range(0, total, CHUNK_SIZE):
    chunk = texts[i:i + CHUNK_SIZE]
    vecs  = embedding_provider.embed_batch(chunk)
    all_embeddings.extend(vecs)

    done    = min(i + CHUNK_SIZE, total)
    elapsed = time.time() - start
    eta     = (elapsed / done) * (total - done)
    print(f'[{done}/{total} | {done/total*100:.1f}%] eta={eta/60:.1f}m  dim={len(vecs[0])}')

processed_df['embedding'] = all_embeddings
print(f'\n[DONE] {total} embeddings — dim={len(all_embeddings[0])}')

## 6. Salvar em data/

In [ ]:
import json

output_path = DATA_DIR / 'processed_candidates.json'

records = processed_df.to_dict(orient='records')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f'Salvo em {output_path}')
print(f'Registros : {len(records)}')
print(f'Colunas   : {list(processed_df.columns)}')